In [1]:
!pip install --upgrade transformers==4.43.3 accelerate torch torchaudio soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 5.8 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 132.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 316.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 180.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 73.7 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 91.5 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 214.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 137.8 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 186.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 78.2 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 133.9 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 114.1 MB/s  0:00:00


In [3]:
# !pip install --upgrade transformers==4.43.3 accelerate torch torchaudio soundfile librosa

import os, re, sys, time, json, glob, math, hashlib, datetime, logging
from collections import defaultdict

import torch
import soundfile as sf
from transformers import pipeline

# =========================
# ======= CONFIG ==========
# =========================
AUDIO_ROOT = "/work/FPSC/wav_speeches"      # 89k WAVs live here
OUTPUT_DIR = "/work/FPSC/output"            # meeting-level JSONLs + logs here

# Choose ONE
# MODEL_ID = "davidilag/wav2vec2-xls-r-300m-faroese-100h-60-epochs_20250122"
MODEL_ID = "davidilag/wav2vec2-xls-r-300m-cpt-1000h_faroese-cp_best-faroese-100h-60-epochs_run8_2025-09-24"

# Inference parameters
CHUNK_LENGTH_S = 30
STRIDE_LENGTH_S = 5
RETURN_TIMESTAMPS = "word"
DECODE_TASK = "transcribe"

# Whisper-only (for schema parity; not used by CTC)
TEMPERATURE = 0.0
CONDITION_ON_PREV_TOKENS = False

# File discovery
GLOB_PATTERN = "*.wav"
ID_RE = re.compile(r"^(M\d+)_S(\d{4})\.wav$", re.IGNORECASE)

# =========================
# ===== UTILITIES =========
# =========================
def short_model_name(model_id: str) -> str:
    # Per your instruction, both map to "wav2vec2-cpt"
    explicit = {
        "davidilag/wav2vec2-xls-r-300m-cpt-1000h_faroese-cp_best-faroese-100h-60-epochs_run8_2025-09-24": "wav2vec2-cpt-fo",
        "davidilag/wav2vec2-xls-r-300m-faroese-100h-60-epochs_20250122": "wav2vec2-fo",
    }
    return explicit.get(model_id, "wav2vec2-cpt")

def sha256_file(path: str, buf_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf_size), b""):
            h.update(chunk)
    return h.hexdigest()

def s_to_hhmmss_ms(s: float | None) -> str | None:
    if s is None: return None
    if s < 0: s = 0.0
    ms = int(round((s - int(s)) * 1000))
    total = int(s)
    hh = total // 3600
    mm = (total % 3600) // 60
    ss = total % 60
    return f"{hh:02d}:{mm:02d}:{ss:02d}.{ms:03d}"

def setup_logger(base_dir: str, short_name: str) -> str:
    os.makedirs(base_dir, exist_ok=True)
    stamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
    log_path = os.path.join(base_dir, f"{short_name}_{stamp}.log")
    # In notebooks: stream to stdout + file
    logger = logging.getLogger()
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)
    logging.info("Logging to %s", log_path)
    return log_path  # <-- fixed (no stray brace)

def list_meeting_files(audio_root: str) -> dict[str, list[str]]:
    files = glob.glob(os.path.join(audio_root, GLOB_PATTERN))
    meetings = defaultdict(list)
    bad = 0
    for fp in files:
        name = os.path.basename(fp)
        m = ID_RE.match(name)
        if not m:
            bad += 1
            continue
        meetings[m.group(1)].append(fp)  # "M1"
    if bad:
        logging.warning("Skipped %d files that did not match pattern %s", bad, ID_RE.pattern)
    return meetings

def sort_key_by_speech_id(path: str) -> int:
    name = os.path.basename(path)
    m = ID_RE.match(name)
    return int(m.group(2)) if m else 10**9

# =========================
# ======= MAIN RUN ========
# =========================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    smn = short_model_name(MODEL_ID)
    log_file = setup_logger(OUTPUT_DIR, smn)
    logging.info("Model: %s (short: %s)", MODEL_ID, smn)
    logging.info("Audio root: %s | Output dir: %s", AUDIO_ROOT, OUTPUT_DIR)

    # Device
    if torch.cuda.is_available():
        device_str = f"cuda:{torch.cuda.current_device()}"
        device_idx = 0
        torch.set_float32_matmul_precision("high")
        logging.info("Using GPU: %s", torch.cuda.get_device_name(0))
    else:
        device_str = "cpu"
        device_idx = -1
        logging.info("Using CPU")

    # Pipeline
    asr = pipeline(
        task="automatic-speech-recognition",
        model=MODEL_ID,
        device=device_idx,
    )

    # Discover files
    meetings = list_meeting_files(AUDIO_ROOT)
    meeting_ids = sorted(meetings.keys(), key=lambda x: int(x[1:]) if x[1:].isdigit() else 10**9)
    logging.info("Found %d meetings", len(meeting_ids))

    # Stats
    total_files = ok_files = fail_files = total_words = 0
    total_audio_seconds = total_wall_seconds = 0.0
    t_run0 = time.time()

    # Process per meeting
    for idx, mtg in enumerate(meeting_ids, start=1):
        wavs = sorted(meetings[mtg], key=sort_key_by_speech_id)
        if not wavs:
            continue

        meeting_out = os.path.join(OUTPUT_DIR, f"{mtg}_{smn}.jsonl")

        # ---- Notebook progress line per meeting ----
        print(f"[{idx}/{len(meeting_ids)}] Meeting {mtg}: {len(wavs)} speeches -> {meeting_out}")
        sys.stdout.flush()

        logging.info("=== Meeting %s | %d speeches -> %s", mtg, len(wavs), meeting_out)

        with open(meeting_out, "a", encoding="utf-8") as fout:
            for wav_path in wavs:
                total_files += 1
                name = os.path.basename(wav_path)
                m = ID_RE.match(name)
                if not m:
                    fail_files += 1
                    logging.error("Name parse failed (skipping): %s", name)
                    continue

                mtg_id, speech_num = m.group(1), m.group(2)
                audio_id = f"{mtg_id}_S{speech_num}"

                try:
                    info = sf.info(wav_path)
                    audio_duration = float(info.duration)
                    audio_sr = int(info.samplerate)
                    audio_channels = int(info.channels)
                    audio_fmt = info.format
                    audio_subtype = info.subtype

                    t0 = time.time()
                    result = asr(
                        wav_path,
                        return_timestamps=RETURN_TIMESTAMPS,
                        chunk_length_s=CHUNK_LENGTH_S,
                        stride_length_s=STRIDE_LENGTH_S,
                    )
                    wall = time.time() - t0
                    rtf = wall / audio_duration if audio_duration > 0 else None

                    full_text = (result.get("text") or "").strip()
                    chunks = result.get("chunks", []) or []

                    words_out = []
                    for ch in chunks:
                        wtxt = ch.get("text", "")
                        ts = ch.get("timestamp", None)
                        if isinstance(ts, (list, tuple)) and len(ts) == 2:
                            start_s = float(ts[0]) if ts[0] is not None else None
                            end_s = float(ts[1]) if ts[1] is not None else None
                        else:
                            start_s = end_s = None
                        words_out.append({
                            "text": wtxt,
                            "start_s": start_s,
                            "end_s": end_s,
                            "start_hhmmss": s_to_hhmmss_ms(start_s),
                            "end_hhmmss": s_to_hhmmss_ms(end_s),
                        })

                    record = {
                        "schema_version": "1.0",
                        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                        "framework": "transformers",
                        "model": MODEL_ID,
                        "audio_id": audio_id,  # <— placed right after "model"
                        "inference_device": device_str,
                        "decode_params": {
                            "chunk_length_s": CHUNK_LENGTH_S,
                            "stride_length_s": STRIDE_LENGTH_S,
                            "return_timestamps": RETURN_TIMESTAMPS,
                            "temperature": TEMPERATURE,
                            "condition_on_prev_tokens": CONDITION_ON_PREV_TOKENS,
                            "task": DECODE_TASK,
                        },
                        "timing": {
                            "wall_time_s": wall,
                            "audio_duration_s": audio_duration,
                            "rtf": rtf,
                        },
                        "audio": {
                            "source_path": wav_path,
                            "source_sha256": sha256_file(wav_path),
                            "source_meta": {
                                "sample_rate_hz": audio_sr,
                                "channels": audio_channels,
                                "duration_s": audio_duration,
                                "subtype": audio_subtype,
                                "format": audio_fmt,
                            },
                        },
                        "transcript": {
                            "full_text": full_text,
                            "num_words": len(words_out),
                        },
                        "words": words_out,
                    }

                    fout.write(json.dumps(record, ensure_ascii=False) + "\n")
                    fout.flush()

                    ok_files += 1
                    total_audio_seconds += audio_duration
                    total_wall_seconds += wall
                    total_words += len(words_out)

                except Exception as e:
                    fail_files += 1
                    logging.exception("FAIL %-13s | %s", audio_id, e)

    # Final stats
    t_run = time.time() - t_run0
    avg_rtf = (total_wall_seconds / total_audio_seconds) if total_audio_seconds > 0 else float("nan")
    total_hours = total_audio_seconds / 3600.0

    logging.info("========== RUN COMPLETE ==========")
    logging.info("Model: %s (%s)", MODEL_ID, smn)
    logging.info("Meetings processed: %d", len(meeting_ids))
    logging.info("Files total/ok/fail: %d / %d / %d", total_files, ok_files, fail_files)
    logging.info("Audio time: %.2f hours", total_hours)
    logging.info("Wall time (sum): %.2f s", total_wall_seconds)
    logging.info("Average RTF: %.3f", avg_rtf if not math.isnan(avg_rtf) else float("nan"))
    logging.info("Total words: %d", total_words)
    logging.info("Elapsed run time: %.2f s", t_run)
    logging.info("Outputs: %s/M*_*.jsonl", OUTPUT_DIR)

    print("\n=== SUMMARY ===")
    print(f"Model: {MODEL_ID} ({smn})")
    print(f"Meetings processed: {len(meeting_ids)}")
    print(f"Files total/ok/fail: {total_files}/{ok_files}/{fail_files}")
    print(f"Audio time: {total_hours:.2f} h")
    print(f"Wall time (sum): {total_wall_seconds:.1f} s")
    print(f"Average RTF: {avg_rtf:.3f}" if not math.isnan(avg_rtf) else "Average RTF: n/a")
    print(f"Total words: {total_words}")
    print(f"Log file is in {OUTPUT_DIR} (see timestamped file with short model name)")
    print(f"Outputs: {OUTPUT_DIR}/M*_*.jsonl")

# Run in notebook
main()


2025-10-10 22:41:33,457 | INFO | Logging to /work/FPSC/output/wav2vec2-cpt-fo_20251010-204133.log
2025-10-10 22:41:33,458 | INFO | Model: davidilag/wav2vec2-xls-r-300m-cpt-1000h_faroese-cp_best-faroese-100h-60-epochs_run8_2025-09-24 (short: wav2vec2-cpt-fo)
2025-10-10 22:41:33,458 | INFO | Audio root: /work/FPSC/wav_speeches | Output dir: /work/FPSC/output


/tmp/ipykernel_1059/3595155300.py:64: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")


2025-10-10 22:41:33,966 | INFO | Using GPU: NVIDIA H100 80GB HBM3


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

2025-10-10 22:41:42,049 | INFO | Found 368 meetings
[1/368] Meeting M1: 3 speeches -> /work/FPSC/output/M1_wav2vec2-cpt-fo.jsonl
2025-10-10 22:41:42,050 | INFO | === Meeting M1 | 3 speeches -> /work/FPSC/output/M1_wav2vec2-cpt-fo.jsonl
[2/368] Meeting M2: 1039 speeches -> /work/FPSC/output/M2_wav2vec2-cpt-fo.jsonl
2025-10-10 22:41:46,553 | INFO | === Meeting M2 | 1039 speeches -> /work/FPSC/output/M2_wav2vec2-cpt-fo.jsonl


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[3/368] Meeting M4: 4 speeches -> /work/FPSC/output/M4_wav2vec2-cpt-fo.jsonl
2025-10-10 22:46:59,525 | INFO | === Meeting M4 | 4 speeches -> /work/FPSC/output/M4_wav2vec2-cpt-fo.jsonl
[4/368] Meeting M5: 147 speeches -> /work/FPSC/output/M5_wav2vec2-cpt-fo.jsonl
2025-10-10 22:47:03,114 | INFO | === Meeting M5 | 147 speeches -> /work/FPSC/output/M5_wav2vec2-cpt-fo.jsonl
[5/368] Meeting M6: 1 speeches -> /work/FPSC/output/M6_wav2vec2-cpt-fo.jsonl
2025-10-10 22:47:48,081 | INFO | === Meeting M6 | 1 speeches -> /work/FPSC/output/M6_wav2vec2-cpt-fo.jsonl
[6/368] Meeting M7: 20 speeches -> /work/FPSC/output/M7_wav2vec2-cpt-fo.jsonl
2025-10-10 22:47:48,683 | INFO | === Meeting M7 | 20 speeches -> /work/FPSC/output/M7_wav2vec2-cpt-fo.jsonl
[7/368] Meeting M8: 3 speeches -> /work/FPSC/output/M8_wav2vec2-cpt-fo.jsonl
2025-10-10 22:47:56,003 | INFO | === Meeting M8 | 3 speeches -> /work/FPSC/output/M8_wav2vec2-cpt-fo.jsonl
[8/368] Meeting M11: 219 speeches -> /work/FPSC/output/M11_wav2vec2-cpt-fo